# Train an emergency-vehicle detector (YOLOv8)

The standard YOLOv8 and Faster R-CNN models are trained on COCO, which has **no ambulance, fire truck or police class**. This notebook fine-tunes YOLOv8 so the app can detect emergency vehicles.

**Dataset:** [EV vs NEV Annotated Vehicle Image Dataset](https://www.kaggle.com/datasets/rohitshitole96/ev-vs-nev-annotated-vehicle-image-dataset) by Rohit Shitole (CC BY-SA 4.0): traffic-camera images with YOLO boxes for emergency vehicles (EV) and non-emergency vehicles (NEV).

**A data-leakage check comes first.** All images are named `aug_…`, so they look like augmented copies (flips, brightness changes) of fewer original photos, split into train and val at random. If copies of the same photo are in both sets, the validation score is too optimistic. Section 2 measures this and builds a clean split where near-duplicates stay together.

**Kaggle settings:** Accelerator → **GPU T4 x2**, Internet → **On**, and add the dataset above as input. Run time: about 45–60 minutes.

In [ ]:
!pip install -q ultralytics
import glob, json, os, random, shutil
from pathlib import Path

import numpy as np
import yaml

src_yaml = sorted(glob.glob("/kaggle/input/**/final_split2/data.yaml", recursive=True))[0]
SRC = Path(src_yaml).parent
print("dataset folder:", SRC)
print(open(src_yaml).read())

## 1. Look at the data

In [ ]:
info = yaml.safe_load(open(src_yaml))
names = info["names"] if isinstance(info["names"], list) else [info["names"][k] for k in sorted(info["names"])]
print("classes:", names)

def is_emergency_name(n):
    n = str(n).lower()
    return n in ("ev", "emergency", "emergency vehicle", "emergency_vehicle") or ("emergency" in n and "non" not in n)

# Clear names for the app: it treats a class as an emergency vehicle when the name contains "emergency".
NEW_NAMES = ["emergency vehicle" if is_emergency_name(n) else "other vehicle" for n in names]
print("renamed to:", NEW_NAMES)
assert NEW_NAMES.count("emergency vehicle") == 1, "check the class names above"

images = sorted(glob.glob(f"{SRC}/train/images/*")) + sorted(glob.glob(f"{SRC}/val/images/*"))
orig_split = {p: ("val" if "/val/" in p else "train") for p in images}

def label_path(img):
    return str(Path(img).parent.parent / "labels" / (Path(img).stem + ".txt"))

def read_labels(lp):
    """Valid YOLO lines only: class x y w h, class in range, coordinates between 0 and 1."""
    good, bad = [], []
    if not os.path.exists(lp):
        return good, bad
    for line in open(lp, errors="ignore").read().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        try:
            cls, xywh = int(parts[0]), [float(v) for v in parts[1:]]
            ok = len(xywh) == 4 and 0 <= cls < len(names) and all(0 <= v <= 1 for v in xywh)
        except ValueError:
            ok = False
        (good if ok else bad).append(line)
    return good, bad

box_counts, bad_lines, bad_files, examples = [0] * len(names), 0, 0, []
for p in images:
    good, bad = read_labels(label_path(p))
    for line in good:
        box_counts[int(line.split()[0])] += 1
    if bad:
        bad_lines += len(bad)
        bad_files += 1
        examples += bad[:1]
print(len(images), "images:", sum(v == "train" for v in orig_split.values()), "train,",
      sum(v == "val" for v in orig_split.values()), "val")
print("boxes per class:", dict(zip(NEW_NAMES, box_counts)))
print(f"invalid label lines: {bad_lines} in {bad_files} files (dropped). Examples:", examples[:5])

## 2. Data-leakage check: near-duplicate images across train and val
Each image is turned into a 512-number feature vector by a ResNet-18 pretrained on ImageNet. Two images are near-duplicates when their cosine similarity is very high (the flipped version of each image is compared too, because flipping is a common augmentation).

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet18_Weights, resnet18

device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = torch.nn.Identity()
backbone = backbone.eval().to(device)
prep = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

feats, feats_flip = [], []
with torch.no_grad():
    for i in range(0, len(images), 64):
        batch = torch.stack([prep(Image.open(p).convert("RGB")) for p in images[i:i + 64]]).to(device)
        for out, x in ((feats, batch), (feats_flip, torch.flip(batch, dims=[3]))):
            f = backbone(x)
            out.append(torch.nn.functional.normalize(f, dim=1).cpu())
F, Ff = torch.cat(feats), torch.cat(feats_flip)
sim = torch.maximum(F @ F.T, F @ Ff.T).numpy()
np.fill_diagonal(sim, -1)
print("similarity matrix:", sim.shape)

In [ ]:
train_idx = [i for i, p in enumerate(images) if orig_split[p] == "train"]
val_idx = [i for i, p in enumerate(images) if orig_split[p] == "val"]
nearest = sim[np.ix_(val_idx, train_idx)].max(axis=1)
leak = {}
for t in (0.85, 0.90, 0.92, 0.95, 0.98):
    leak[str(t)] = round(float((nearest >= t).mean()), 4)
    print(f"val images with a train image at similarity >= {t}: {leak[str(t)]:.1%}")

Example pairs at different similarity levels (left: val image, right: its most similar train image). This shows where "the same photo" ends and "a different photo" begins.

In [ ]:
import matplotlib.pyplot as plt

bins = [(0.98, 1.01), (0.95, 0.98), (0.92, 0.95), (0.88, 0.92), (0.80, 0.88)]
rng = random.Random(0)
fig, axes = plt.subplots(len(bins), 4, figsize=(14, 3.2 * len(bins)))
for row, (lo, hi) in enumerate(bins):
    cand = [k for k, s in enumerate(nearest) if lo <= s < hi]
    picks = rng.sample(cand, min(2, len(cand)))
    for col in range(4):
        axes[row, col].axis("off")
    for j, k in enumerate(picks):
        v = val_idx[k]
        t = train_idx[int(sim[v, train_idx].argmax())]
        axes[row, 2 * j].imshow(Image.open(images[v]))
        axes[row, 2 * j].set_title(f"val  sim={nearest[k]:.3f}", fontsize=9)
        axes[row, 2 * j + 1].imshow(Image.open(images[t]))
        axes[row, 2 * j + 1].set_title("nearest train", fontsize=9)
plt.tight_layout()
plt.savefig("/kaggle/working/near_duplicate_examples.png", dpi=80)
plt.show()

### Build a clean split
Images connected by similarity ≥ `THRESHOLD` are put in one group, and whole groups go to train, val or test (70 / 15 / 15). No near-duplicate can then appear on both sides.

In [ ]:
THRESHOLD = 0.92

parent = list(range(len(images)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in zip(*np.where(np.triu(sim, 1) >= THRESHOLD)):
    parent[find(a)] = find(b)
groups = {}
for i in range(len(images)):
    groups.setdefault(find(i), []).append(i)
group_list = list(groups.values())
random.Random(42).shuffle(group_list)
print(len(group_list), "groups; largest group:", max(map(len, group_list)), "images")

split_of, total = {}, len(images)
counts = {"train": 0, "val": 0, "test": 0}
for g in group_list:
    target = "train" if counts["train"] < 0.70 * total else ("val" if counts["val"] < 0.15 * total else "test")
    for i in g:
        split_of[i] = target
    counts[target] += len(g)
print("clean split:", counts)

CLEAN = Path("/kaggle/working/ev_clean")
shutil.rmtree(CLEAN, ignore_errors=True)
for i, p in enumerate(images):
    s = split_of[i]
    (CLEAN / s / "images").mkdir(parents=True, exist_ok=True)
    (CLEAN / s / "labels").mkdir(parents=True, exist_ok=True)
    os.symlink(p, CLEAN / s / "images" / Path(p).name)
    good, _ = read_labels(label_path(p))  # cleaned label file: valid lines only
    (CLEAN / s / "labels" / (Path(p).stem + ".txt")).write_text("\n".join(good) + ("\n" if good else ""))
data_yaml = str(CLEAN / "data.yaml")
yaml.safe_dump({"path": str(CLEAN), "train": "train/images", "val": "val/images", "test": "test/images",
                "names": NEW_NAMES}, open(data_yaml, "w"))
# check: the leak inside the clean split
cv = [i for i in range(total) if split_of[i] != "train"]
ct = [i for i in range(total) if split_of[i] == "train"]
print("clean val+test images with a train image at similarity >=", THRESHOLD, ":",
      f"{(sim[np.ix_(cv, ct)].max(axis=1) >= THRESHOLD).mean():.1%}")

## 3. Train YOLOv8s on the clean split
Fine-tuned from COCO weights for up to 60 epochs; training stops early if validation mAP does not improve for 15 epochs. The best epoch is chosen on **val**; the final score is on **test**, which is used only once.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(data=data_yaml, epochs=60, imgsz=640, batch=16, patience=15, device=0, seed=0,
            project="/kaggle/working/runs", name="emergency", exist_ok=True, plots=True)

## 4. Evaluate on the clean test set (never seen in training or model selection)

In [ ]:
best = YOLO("/kaggle/working/runs/emergency/weights/best.pt")
m = best.val(data=data_yaml, split="test", imgsz=640, plots=True,
             project="/kaggle/working/runs", name="emergency_test", exist_ok=True)
per_class = {best.names[int(c)]: {"precision": round(float(p), 4), "recall": round(float(r), 4),
                                  "mAP50": round(float(a50), 4), "mAP50_95": round(float(a), 4)}
             for c, p, r, a50, a in zip(m.box.ap_class_index, m.box.p, m.box.r, m.box.ap50, m.box.ap)}
results = {
    "dataset": "rohitshitole96/ev-vs-nev-annotated-vehicle-image-dataset (CC BY-SA 4.0)",
    "images": total, "original_split": {"train": len(train_idx), "val": len(val_idx)},
    "leak_val_with_near_duplicate_in_train": leak, "threshold": THRESHOLD,
    "clean_split": counts, "groups": len(group_list),
    "invalid_label_lines_dropped": bad_lines, "label_files_with_invalid_lines": bad_files,
    "model": "YOLOv8s fine-tuned from COCO",
    "test": {"precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
             "mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4)},
    "test_per_class": per_class,
}
import pandas as pd
hist = pd.read_csv("/kaggle/working/runs/emergency/results.csv")
hist.columns = [c.strip() for c in hist.columns]
results["epochs_trained"] = int(len(hist))
results["best_epoch"] = int(hist["metrics/mAP50-95(B)"].idxmax()) + 1
print(json.dumps(results, indent=2))
json.dump(results, open("/kaggle/working/emergency_metrics.json", "w"), indent=2)

In [ ]:
from IPython.display import Image as Show, display
for path in ("/kaggle/working/runs/emergency/results.png",
             "/kaggle/working/runs/emergency_test/confusion_matrix_normalized.png",
             "/kaggle/working/runs/emergency_test/val_batch0_pred.jpg"):
    if os.path.exists(path):
        display(Show(path, width=900))
shutil.copy("/kaggle/working/runs/emergency/weights/best.pt", "/kaggle/working/emergency_yolov8s.pt")
shutil.rmtree(CLEAN)  # image symlinks and small label files; keeps the output small
print("Saved emergency_yolov8s.pt and emergency_metrics.json")

## Observations
*(Write 3–4 points after the run: how big the leak was, the test mAP for emergency vehicles, which mistakes you see in the prediction images.)*

## Use it in the app
Download `emergency_yolov8s.pt` from the **Output** tab, put it in the app's `models/` folder and restart the app. Choose it under **Emergency-vehicle model**: the normal YOLOv8 model keeps counting all vehicles, and this model adds emergency detection, alerts and signal priority.